# 辞書やルールで固有表現を追加する (EntityRuler)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/16_ner/entity_ruler.ipynb)

NLP クックブック 16-05 ／ 固有表現抽出

統計モデルが取りこぼす製品名・専門用語などを、手持ちの辞書で確実に固有表現として認識させたい。

In [1]:
!pip install -q ginza ja_ginza

In [2]:
import spacy

nlp = spacy.load("ja_ginza")
text = "山田研究室では、GECToRとBARTを使って文法誤り訂正の実験をした。"
print("追加前:", [(e.text, e.label_) for e in nlp(text).ents])

ruler = nlp.add_pipe("entity_ruler", before="ner")  # ner の前に置くと辞書が優先される
ruler.add_patterns([
    {"label": "MODEL", "pattern": "GECToR"},
    {"label": "MODEL", "pattern": "BART"},
    {"label": "TASK", "pattern": [{"ORTH": "文法"}, {"ORTH": "誤り"}, {"ORTH": "訂正"}]},  # トークン列のパターン
])
print("追加後:", [(e.text, e.label_) for e in nlp(text).ents])

追加前: [('山田研究室', 'Person')]
追加後: [('山田研究室', 'Person'), ('GECToR', 'MODEL'), ('BART', 'MODEL'), ('文法誤り訂正', 'TASK')]


In [3]:
# パターンを JSONL に保存・読み込み
ruler.to_disk("patterns.jsonl")
print(open("patterns.jsonl", encoding="utf-8").read())

{"label":"TASK","pattern":[{"ORTH":"\u6587\u6cd5"},{"ORTH":"\u8aa4\u308a"},{"ORTH":"\u8a02\u6b63"}]}
{"label":"MODEL","pattern":"GECToR"}
{"label":"MODEL","pattern":"BART"}



- 文字列パターンはトークナイザの分割と一致する必要がある。合わないときは、トークン列を確認してトークンパターンで書く。
- `after="ner"` に置くと統計モデルの結果が優先され、重ならない部分だけ辞書で補う動作になる。